<div style="border-left:4px solid #3498db;padding-left:15px;margin-bottom:20px;">
  <h1 style="margin-top:0;">🏭☀️ 01. Energy Forecasting: Time-Series & ML Benchmarks</h1>
  <p style="font-size:1.1em;color:#888;">Predicting the next quarter-hour of machine demand and solar generation, with honest tests before smarter scheduling.</p>
</div>

### 🎯 Main Goals & Business Value
* **⏱️ Forecast the next interval:** Predict average machine power and solar power separately. Good short-term forecasts help a scheduler prepare for changes.
* **📏 Beat a simple answer:** Compare learned models with the latest reading, recent average and the same time yesterday or last week.
* **🧪 Choose fairly:** Compare models using earlier time periods, tune the strongest ML candidate, then evaluate the chosen approach on untouched later data. XGBoost gets the same chance as the others.
* **🛡️ Keep the evidence clear:** Missing targets are excluded, missing features are handled inside each training pipeline, and data from the future never enters a feature.

### 🛠️ Core Tools & Approach
* **Data engineering:** `Polars` reads the small ETL model exports and quarter-hour reference files. We do not load the full five-second telemetry.
* **Time-series methods:** Persistence, moving averages, seasonal reuse and autoregressive Ridge provide understandable benchmarks.
* **Machine learning:** Ridge, Random Forest, histogram gradient boosting and XGBoost compete using the same prediction rows.
* **Preprocessing:** Training-only median imputation, missing-value indicators, scaling for linear models and weekday encoding.
* **Visualization:** `Plotly` charts explain coverage, model errors, actual versus predicted power and where the selected model struggles.

---

### 🗄️ Data Structure
The notebook reads each system's `ml.parquet`, `eda_15min.parquet` and `quality.json` through the published ETL manifest.

| Domain | Examples | Purpose |
| :--- | :--- | :--- |
| 🏭 Machine target | `target_power_kw` for `TEC_48S` | Next quarter-hour average total active power. Small signed readings are preserved. |
| ☀️ Solar target | `target_power_kw` for `IPE_PV` | Next quarter-hour average AC generation. |
| ⏱️ Forecast timing | `WsDateTime`, `Target_Start`, `Target_End` | Issuing time and the interval being predicted. |
| 📈 Known history | Latest power, rolling statistics, exact past-day and past-week values | Describe what was observed before issuing the forecast. |
| 🌤️ External context | Previous-hour temperature and published day-ahead price | Historical context available under the stated release-time assumptions. |
| 🧪 Evaluation | `split` | Preserve the ETL's train, validation and test periods. |

> **Next Step Alignment:** This is a rolling next-quarter-hour forecast, not a 96-step next-day forecast. Test-time forecasts may use newly observed past readings as time advances. Later scheduling must respect this horizon. Predicted machine demand is not automatically grid imports, and a shared machine–solar connection still needs confirmation.

### 🛠️ 1. Setup and sensible limits
Run this notebook inside the project with your ML environment selected. If needed, install `polars`, `pyarrow`, `numpy`, `scikit-learn`, `xgboost`, `plotly`, `joblib`, `threadpoolctl` and `psutil` in that environment.

Two worker threads and one model at a time keep memory use predictable. Tune 12 candidates by default; lower this number for a quick check. Logs go to `logs/model_training/`. No automatic package installation or ETL run takes place.

In [15]:
import os
THREADS = 2
for variable in ("POLARS_MAX_THREADS", "OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(variable, str(THREADS))
from pathlib import Path
from datetime import datetime, timezone
from html import escape
import sys, json, time, hashlib, platform, importlib.metadata as metadata
from datetime import timedelta
from zoneinfo import ZoneInfo
import numpy as np
import polars as pl
import pyarrow.parquet as pq
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, HTML, Markdown
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.model_selection import ParameterSampler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.inspection import permutation_importance
from threadpoolctl import threadpool_limits
import joblib
try:
    from xgboost import XGBRegressor
except ImportError as error:
    raise ImportError("Install XGBoost in the selected notebook environment: python -m pip install xgboost") from error

SEED, TIMEZONE = 42, "Europe/Berlin"
SYSTEMS = ("TEC_48S", "IPE_PV")
TUNE_TRIALS, CV_FOLDS = 12, 3
MIN_TRAIN, MIN_EVALUATION = 128, 64
MAX_ML_ROWS = 120_000
SAVE_ARTIFACTS = True
ENABLE_MLFLOW = False
BLUE, ORANGE, PURPLE, RED = "#2563eb", "#f59e0b", "#7c3aed", "#dc2626"
PROJECT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "etl_config.json").is_file()), None)
if PROJECT is None:
    raise FileNotFoundError("Run inside the GreenForge AI project containing etl_config.json.")
if str(PROJECT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT / "src"))
from greenforge_ai.features.interaction_features import artifact
from greenforge_ai.utils.logger import setup_logging, get_logger, stage
setup_logging(PROJECT / "logs", new_run=True)
log = get_logger("forecasting_benchmarks", channel="training")
import logging

channel = logging.getLogger("greenforge.training")
for handler in channel.handlers[:]:
    if isinstance(handler, logging.StreamHandler) and not isinstance(
        handler, logging.FileHandler
    ):
        channel.removeHandler(handler)
        handler.close()
        
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%fZ")
OUTPUT = PROJECT / "models" / "forecasting" / RUN_ID
settings = dict(seed=SEED, threads=THREADS, cv_folds=CV_FOLDS, tune_trials=TUNE_TRIALS,
                task="rolling_next_quarter_hour", selection_metric="MAE_kW")
log.info("Forecasting run=%s settings=%s", RUN_ID, settings)

In [16]:
def report(text):
    display(Markdown("> **💡 What this tells us**\n>\n" + "\n".join("> " + line for line in text.splitlines())))

def cards(items):
    content = "".join(
        '<div style="padding:18px;border:1px solid #e2e8f0;border-radius:12px;background:white;color:#334155">'
        f'<div style="font-size:13px;color:#64748b">{escape(label)}</div>'
        f'<div style="font-size:24px;font-weight:700;margin:9px 0">{escape(value)}</div>'
        f'<div style="font-size:12px;line-height:1.5">{escape(note)}</div></div>'
        for label, value, note in items)
    display(HTML('<div style="display:grid;grid-template-columns:repeat(3,minmax(0,1fr));gap:12px;margin:14px 0">' + content + '</div>'))

def polish(fig, title, subtitle, height=450):
    fig.update_layout(template="plotly_white", height=height,
        title=dict(text=title + "<br><sup>" + subtitle + "</sup>", x=.02, font=dict(size=20)),
        font=dict(family="Arial", size=13, color="#334155"), margin=dict(l=75,r=35,t=115,b=65),
        legend=dict(orientation="h", y=1.02, yanchor="bottom"), hovermode="x unified")
    fig.update_xaxes(showgrid=False, automargin=True)
    fig.update_yaxes(gridcolor="#eef2f7", automargin=True)
    fig.show(config=dict(displaylogo=False, responsive=True))

def clock(series):
    # Offset prevents repeated autumn-clock hours becoming identical labels.
    return series.dt.convert_time_zone(TIMEZONE).dt.strftime("%Y-%m-%d %H:%M:%S%z").to_list()

def scores(actual, predicted):
    actual, predicted = np.asarray(actual), np.asarray(predicted)
    if actual.shape != predicted.shape or not np.isfinite(predicted).all():
        raise ValueError("Predictions must be finite and match the target rows.")
    denominator = float(np.abs(actual).sum())
    return dict(mae_kw=float(mean_absolute_error(actual, predicted)),
                rmse_kw=float(np.sqrt(mean_squared_error(actual, predicted))),
                bias_kw=float(np.mean(predicted - actual)),
                wape_pct=100 * float(np.abs(predicted - actual).sum()) / denominator if denominator else None)

SOURCE_NUMERIC = ["power_last_15m_kw", "power_previous_15m_kw", "power_history_mean_kw",
                  "power_history_std_kw", "previous_hour_temperature_c", "target_day_ahead_price_eur_mwh"]
NUMERIC = SOURCE_NUMERIC + ["hour_sin", "hour_cos", "week_sin", "week_cos", "month_sin", "month_cos",
                           "weekend", "power_change_kw", "history_deviation_kw", "power_day_ago_kw", "power_week_ago_kw"]
FEATURES = NUMERIC + ["weekday"]
AR_FEATURES = ["power_last_15m_kw", "power_previous_15m_kw", "power_history_mean_kw",
               "power_history_std_kw", "power_day_ago_kw", "power_week_ago_kw"]
TIME_COLUMNS = ["WsDateTime", "Feature_Window_Start", "Target_Start", "Target_End"]

### 📥 2. Read the published datasets and verify the forecast contract
The quality reports must approve the selected targets. A generic `ready_for_review` status for auxiliary channels does not override the target-specific checks.

We verify timestamps, quarter-hour target duration, chronological split boundaries and finite targets. Features may be missing, but target values are never invented. This experiment keeps the ETL's eligible windows; it cannot recover observations already excluded upstream.

In [17]:
def load_system(system):
    paths = {name: artifact(system, name, PROJECT) for name in ("ml.parquet", "eda_15min.parquet", "quality.json")}
    quality = json.loads(paths["quality.json"].read_text(encoding="utf-8"))
    if quality.get("features", {}).get("ml_status") != "ready" or not quality.get("target_quality_passed", False):
        raise ValueError(f"{system}: target or ML export has not passed the ETL gate.")
    target = "P_total" if system == "TEC_48S" else "AC_ActivePower"
    rule = quality["rules"][target]
    if rule.get("unit") != "W" or rule.get("status", "unverified").startswith("unverified"):
        raise ValueError(f"{system}: verify target units first.")
    metadata_rows = pq.read_metadata(paths["ml.parquet"]).num_rows
    if metadata_rows > MAX_ML_ROWS:
        raise MemoryError(f"{system}: {metadata_rows:,} model rows exceed this notebook's explicit limit; use a batch training workflow.")
    selected_columns = TIME_COLUMNS + SOURCE_NUMERIC + ["target_power_kw", "split"]
    frame = pl.read_parquet(paths["ml.parquet"], columns=selected_columns).sort("WsDateTime")
    for column in TIME_COLUMNS:
        dtype = frame.schema[column]
        if not isinstance(dtype, pl.Datetime) or dtype.time_zone != "UTC":
            raise TypeError(f"{system}/{column}: require UTC datetimes.")
    frame = frame.with_columns(pl.col(TIME_COLUMNS).cast(pl.Datetime("us", "UTC")))
    if frame.select(pl.any_horizontal(pl.col(TIME_COLUMNS).is_null()).any()).item():
        raise ValueError(f"{system}: missing window timestamps.")
    if frame["WsDateTime"].n_unique() != frame.height:
        raise ValueError(f"{system}: duplicate forecast origins.")
    if (frame["WsDateTime"].dt.epoch("us") % 900_000_000 != 0).any():
        raise ValueError(f"{system}: forecast origins are not quarter-hour interval starts.")
    if system == "IPE_PV" and frame.filter(pl.col("target_power_kw") < 0).height:
        raise ValueError("Solar targets must be non-negative; inspect the ETL.")
    invalid_target = (~pl.col("target_power_kw").is_finite()).fill_null(True)
    excluded = frame.filter(invalid_target).height
    frame = frame.filter(~invalid_target)
    if set(frame["split"].unique().to_list()) != {"train", "validation", "test"}:
        raise ValueError(f"{system}: require train, validation and test labels.")
    invalid_clock = frame.filter(
        (pl.col("Target_Start") != pl.col("WsDateTime")) |
        ((pl.col("Target_End") - pl.col("Target_Start")).dt.total_microseconds() != 900_000_000) |
        (pl.col("Feature_Window_Start") >= pl.col("WsDateTime")))
    if invalid_clock.height:
        raise ValueError(f"{system}: this notebook requires the existing next-quarter-hour target contract.")
    for earlier, later in [("train", "validation"), ("validation", "test")]:
        a, b = frame.filter(pl.col("split") == earlier), frame.filter(pl.col("split") == later)
        if a["Target_End"].max() > b["Feature_Window_Start"].min():
            raise ValueError(f"{system}: overlapping split windows ({earlier}/{later}).")
    for column in SOURCE_NUMERIC:
        frame = frame.with_columns(pl.when(pl.col(column).is_finite()).then(pl.col(column)).otherwise(None).alias(column))
    valid_latest = frame["power_last_15m_kw"].count() == frame.height
    if not valid_latest:
        raise ValueError(f"{system}: persistence requires a valid latest power reading for every eligible window.")
    observed = pl.read_parquet(paths["eda_15min.parquet"], columns=["WsDateTime", "power_kw"])
    if observed["WsDateTime"].null_count() or observed["WsDateTime"].n_unique() != observed.height:
        raise ValueError(f"{system}: invalid EDA reference clock.")
    dtype = observed.schema["WsDateTime"]
    if not isinstance(dtype, pl.Datetime) or dtype.time_zone != "UTC":
        raise TypeError(f"{system}: reference timestamps must be UTC.")
    observed = observed.with_columns(pl.col("WsDateTime").cast(pl.Datetime("us", "UTC")))
    log.info("%s ML rows=%d excluded_targets=%d target_rule=%s", system, frame.height, excluded, rule)
    return dict(frame=frame, observed=observed, quality=quality, paths=paths, excluded_targets=excluded)

with stage("load and validate model datasets", logger=log):
    data = {system: load_system(system) for system in SYSTEMS}
for system, entry in data.items():
    frame, quality = entry["frame"], entry["quality"]
    target = "P_total" if system == "TEC_48S" else "AC_ActivePower"
    source = quality["sensors"][target]
    observed_pct = 100 * source["valid"] / quality["rows"]
    display(Markdown(f"### {'🏭' if system == 'TEC_48S' else '☀️'} {system}"))
    counts = {name: frame.filter(pl.col("split") == name).height for name in ("train", "validation", "test")}
    cards([
        ("Eligible model windows", f"{frame.height:,}", "full published ML export · no sampling"),
        ("Train / validation / test", f"{counts['train']:,} / {counts['validation']:,} / {counts['test']:,}", "existing chronological partitions"),
        ("Forecast", "Next 15 minutes", "one average-power prediction · kW"),
        ("Source power coverage", f"{observed_pct:.1f}%", "valid readings across the full five-second grid"),
        ("Imported size", f"{entry['paths']['ml.parquet'].stat().st_size / 1024**2:.2f} MiB", "compressed model file · no five-second load"),
        ("Target status", "ETL gate passed", f"{entry['excluded_targets']:,} invalid exported targets excluded"),
    ])
report("Machine and solar have different usable periods. We compare models within each system, not by raw error across systems. Solar conclusions apply to its observed season, not the full year.")

### 🏭 TEC_48S

### ☀️ IPE_PV

> **💡 What this tells us**
>
> Machine and solar have different usable periods. We compare models within each system, not by raw error across systems. Solar conclusions apply to its observed season, not the full year.

### 🧩 3. Feature engineering: history without future information
We add recent changes, distance from the recent mean, cyclic time features and exact same-time readings one and seven **elapsed UTC days** earlier. Exact joins keep missing periods visible. They do not compress gaps or pretend that the previous row means the previous quarter-hour.

The past-day/week interval must finish before the forecast is issued. Across a Berlin clock change, an elapsed-day feature can represent a different local hour. Weekdays are encoded separately; cyclical features keep adjacent hours close.

Previous-hour temperature follows the ETL's delay assumption. The imminent interval's day-ahead price is assumed already published. Before live deployment, verify source release times; cached historical data alone cannot prove availability.

In [18]:
def engineer(entry):
    frame, observed = entry["frame"], entry["observed"]
    for days, column in [(1, "power_day_ago_kw"), (7, "power_week_ago_kw")]:
        reference = observed.select(
            (pl.col("WsDateTime") + pl.duration(days=days)).alias("Target_Start"),
            (pl.col("WsDateTime") + pl.duration(minutes=15)).alias("reference_end"),
            pl.when(pl.col("power_kw").is_finite()).then(pl.col("power_kw")).otherwise(None).alias(column))
        frame = frame.join(reference, on="Target_Start", how="left", validate="1:1")
        if frame.filter(pl.col("reference_end") > pl.col("WsDateTime")).height:
            raise ValueError("A reference value would come from the future.")
        frame = frame.drop("reference_end")
    local = pl.col("Target_Start").dt.convert_time_zone(TIMEZONE)
    hour = local.dt.hour() + local.dt.minute() / 60
    weekday = local.dt.weekday() - 1
    frame = frame.with_columns(
        (2 * np.pi * hour / 24).sin().alias("hour_sin"), (2 * np.pi * hour / 24).cos().alias("hour_cos"),
        (2 * np.pi * (weekday + hour / 24) / 7).sin().alias("week_sin"),
        (2 * np.pi * (weekday + hour / 24) / 7).cos().alias("week_cos"),
        (2 * np.pi * (local.dt.month() - 1) / 12).sin().alias("month_sin"),
        (2 * np.pi * (local.dt.month() - 1) / 12).cos().alias("month_cos"),
        (weekday >= 5).cast(pl.Float32).alias("weekend"), weekday.cast(pl.Float32).alias("weekday"),
        (pl.col("power_last_15m_kw") - pl.col("power_previous_15m_kw")).alias("power_change_kw"),
        (pl.col("power_last_15m_kw") - pl.col("power_history_mean_kw")).alias("history_deviation_kw"),
    ).with_columns(pl.col(FEATURES + ["target_power_kw"]).cast(pl.Float32)).sort("WsDateTime")
    return frame

with stage("past-only feature engineering", logger=log):
    for system, entry in data.items():
        entry["frame"] = engineer(entry)
        entry.pop("observed")
        entry["splits"] = {name: entry["frame"].filter(pl.col("split") == name)
                           for name in ("train", "validation", "test")}
        for split, frame in entry["splits"].items():
            required = MIN_TRAIN if split == "train" else MIN_EVALUATION
            if frame.height < required:
                raise ValueError(f"{system}/{split}: fewer than {required} eligible rows.")

fig = go.Figure()
for system, entry in data.items():
    train = entry["splits"]["train"]
    fractions = [100 * train[column].null_count() / train.height for column in ["power_day_ago_kw", "power_week_ago_kw"]]
    fig.add_bar(x=["Past day", "Past week"], y=fractions, name=system,
                marker_color=BLUE if system == "TEC_48S" else ORANGE,
                text=[f"{value:.1f}%" for value in fractions], textposition="outside")
fig.update_layout(barmode="group")
fig.update_yaxes(title="Missing training feature values (%)", range=[0,110])
polish(fig, "How much seasonal history is available?", "Exact past intervals · missing history stays missing")
report("A missing historical feature does not mean zero power. Each model fills it using training-only statistics and a missing-value indicator. This does not reconstruct the underlying sensor data or fill a target.")

> **💡 What this tells us**
>
> A missing historical feature does not mean zero power. Each model fills it using training-only statistics and a missing-value indicator. This does not reconstruct the underlying sensor data or fill a target.

### 🧼 4. Preprocessing and honest time-series benchmarks
Linear models use median imputation and standard scaling. Tree models need no scaling. Both use missing indicators and weekday one-hot encoding where calendar features are included. No transformer learns from validation or test data.

The seasonal benchmarks fall back to persistence when their exact historical interval is missing. Their coverage is reported explicitly, so all benchmark scores use the same rows. The autoregressive Ridge model uses only power history; the contextual models can also use time, weather and price.

In [19]:
def arrays(frame, columns=FEATURES):
    return frame.select(columns).to_numpy().astype(np.float32, copy=False)

def make_pipeline(name, columns, params=None):
    numeric_indices = [i for i,c in enumerate(columns) if c != "weekday"]
    scale = name in {"Autoregressive Ridge", "Context Ridge"}
    steps = [("impute", SimpleImputer(strategy="median", add_indicator=True, keep_empty_features=True))]
    if scale:
        steps.append(("scale", StandardScaler()))
    transformers = [("numeric", Pipeline(steps), numeric_indices)]
    if "weekday" in columns:
        transformers.append(("weekday", OneHotEncoder(categories=[list(range(7))], handle_unknown="ignore",
                            sparse_output=False, dtype=np.float32), [columns.index("weekday")]))
    preprocessor = ColumnTransformer(transformers, remainder="drop", sparse_threshold=0)
    if name in {"Autoregressive Ridge", "Context Ridge"}:
        model = Ridge(alpha=10.0)
    elif name == "Random Forest":
        model = RandomForestRegressor(n_estimators=160, max_depth=12, min_samples_leaf=8,
                                      max_features=.8, n_jobs=THREADS, random_state=SEED)
    elif name == "Histogram Boosting":
        model = HistGradientBoostingRegressor(max_iter=180, max_leaf_nodes=15, learning_rate=.05,
                    min_samples_leaf=30, l2_regularization=1., early_stopping=False, random_state=SEED)
    elif name == "XGBoost":
        model = XGBRegressor(n_estimators=250, max_depth=4, learning_rate=.05, min_child_weight=10,
                    subsample=.85, colsample_bytree=.85, reg_lambda=5., objective="reg:squarederror",
                    tree_method="hist", n_jobs=THREADS, random_state=SEED, verbosity=0)
    else:
        raise ValueError(name)
    pipeline = Pipeline([("preprocess", preprocessor), ("model", model)])
    return pipeline.set_params(**(params or {}))

MODEL_COLUMNS = {name: AR_FEATURES if name == "Autoregressive Ridge" else FEATURES
                 for name in ["Autoregressive Ridge", "Context Ridge", "Random Forest", "Histogram Boosting", "XGBoost"]}

def predict_model(model, frame, columns, system):
    predicted = np.asarray(model.predict(arrays(frame, columns)), dtype=float)
    return np.maximum(predicted, 0) if system == "IPE_PV" else predicted

def benchmark(frame, name, system):
    latest = frame["power_last_15m_kw"].to_numpy().astype(float)
    columns = {"Persistence": "power_last_15m_kw", "Recent mean": "power_history_mean_kw",
               "Past day + fallback": "power_day_ago_kw", "Past week + fallback": "power_week_ago_kw"}
    values = frame[columns[name]].to_numpy().astype(float)
    coverage = 100 * np.isfinite(values).mean()
    values = np.where(np.isfinite(values), values, latest)
    return (np.maximum(values, 0) if system == "IPE_PV" else values), coverage

BASELINES = ["Persistence", "Recent mean", "Past day + fallback", "Past week + fallback"]

# Training-only feature audit; target and split columns never enter a model.
for system, entry in data.items():
    train = entry["splits"]["train"]
    all_missing = [column for column in FEATURES if train[column].count() == 0]
    log.info("%s feature_count=%d all_missing_training_features=%s", system, len(FEATURES), all_missing)
report("All-missing training features are retained with a neutral imputation value and cannot supply learned historical variation. Solar predictions are bounded at zero; machine signed power is preserved. We do not cap peaks or delete difficult test examples.")

> **💡 What this tells us**
>
> All-missing training features are retained with a neutral imputation value and cannot supply learned historical variation. Solar predictions are bounded at zero; machine signed power is preserved. We do not cap peaks or delete difficult test examples.

### 🧪 5. Compare models inside the training period
Three expanding time folds test each ML candidate on later parts of the training period. Timestamp-based purging prevents a fitted target interval from crossing the start of the evaluation feature window. Folds are never shuffled.

The same temporal folds determine the tuning candidate. The reserved validation period remains available for the final choice; the test period is not scored or plotted yet. Models train sequentially, with no parallel search or random early-stopping split.

In [20]:
def time_folds(train):
    edges = np.linspace(.4, 1., CV_FOLDS + 1)
    folds = []
    for low, high in zip(edges[:-1], edges[1:]):
        start, stop = int(train.height * low), int(train.height * high)
        evaluation = train.slice(start, stop - start)
        cutoff = evaluation["Feature_Window_Start"].min()
        fitted = train.head(start).filter(pl.col("Target_End") <= cutoff)
        if fitted.height < MIN_TRAIN or evaluation.height < MIN_EVALUATION:
            raise ValueError("Not enough rows for the requested temporal folds.")
        folds.append((fitted, evaluation))
    return folds

def cv_score(system, folds, name, columns, params=None):
    metrics, weights = [], []
    for fold, (fitted, evaluation) in enumerate(folds, 1):
        pipeline = make_pipeline(name, columns, params)
        with threadpool_limits(limits=THREADS):
            pipeline.fit(arrays(fitted, columns), fitted["target_power_kw"].to_numpy())
            prediction = predict_model(pipeline, evaluation, columns, system)
        metrics.append(scores(evaluation["target_power_kw"].to_numpy(), prediction)["mae_kw"])
        weights.append(evaluation.height)
        log.info("%s model=%s fold=%d train=%d evaluation=%d MAE=%.6f", system,name,fold,fitted.height,evaluation.height,metrics[-1])
        del pipeline
    return float(np.average(metrics, weights=weights)), metrics

screen_rows = []
with stage("expanding-fold ML comparison", logger=log):
    for system, entry in data.items():
        entry["folds"] = time_folds(entry["splits"]["train"])
        for name, columns in MODEL_COLUMNS.items():
            started = time.perf_counter()
            mean_mae, fold_errors = cv_score(system, entry["folds"], name, columns)
            screen_rows.append(dict(system=system, model=name, cv_mae_kw=mean_mae,
                                    seconds=time.perf_counter()-started, fold_mae=fold_errors))
screen = pl.DataFrame(screen_rows)
fig = make_subplots(rows=1, cols=len(SYSTEMS), subplot_titles=list(SYSTEMS), horizontal_spacing=.15)
for column, system in enumerate(SYSTEMS, 1):
    ranked = screen.filter(pl.col("system") == system).sort("cv_mae_kw", descending=True)
    fig.add_bar(x=ranked["cv_mae_kw"].to_list(), y=ranked["model"].to_list(), orientation="h",
                marker_color=[PURPLE if name == "XGBoost" else BLUE for name in ranked["model"]], showlegend=False, row=1,col=column)
    fig.update_xaxes(title="Training-fold MAE (kW) · lower is better", row=1,col=column)
polish(fig, "Which ML model handles unseen time periods best?", "Training-only expanding folds · each system has its own error scale", height=470)
report("Purple identifies XGBoost, not a preselected winner. Lower MAE is better. The strongest training-fold candidate for each system now receives a bounded parameter search.")

> **💡 What this tells us**
>
> Purple identifies XGBoost, not a preselected winner. Lower MAE is better. The strongest training-fold candidate for each system now receives a bounded parameter search.

### 🎛️ 6. Tune the strongest ML candidate
Hyperparameters are model settings such as tree depth, learning rate and regularisation. We try a small reproducible sample, using the same training-only temporal folds. The untuned candidate is included, so tuning cannot silently replace it with a worse training-fold score.

We tune one leading ML family per system instead of spending RAM and time on a large search for every model. The validation period will decide whether this tuned candidate improves on the alternatives. If a different ML family leads there, it also receives a training-only parameter search before the final choice is frozen. Default settings remain eligible when they work better.

In [21]:
SEARCH = {
    "Autoregressive Ridge": {"model__alpha": [.01,.1,1.,10.,100.,1000.]},
    "Context Ridge": {"model__alpha": [.01,.1,1.,10.,100.,1000.]},
    "Random Forest": {"model__n_estimators": [120,200], "model__max_depth": [8,14,None],
                      "model__min_samples_leaf": [4,12,24], "model__max_features": [.6,1.]},
    "Histogram Boosting": {"model__max_iter": [120,240], "model__learning_rate": [.03,.07],
                           "model__max_leaf_nodes": [7,15,31], "model__min_samples_leaf": [20,50],
                           "model__l2_regularization": [1.,10.]},
    "XGBoost": {"model__n_estimators": [150,300,450], "model__max_depth": [2,4,6],
                "model__learning_rate": [.03,.07], "model__min_child_weight": [5,20],
                "model__subsample": [.75,1.], "model__colsample_bytree": [.75,1.],
                "model__reg_lambda": [1.,10.], "model__reg_alpha": [0.,1.]},
}
tuning_rows = []
with stage("bounded temporal hyperparameter search", logger=log):
    for system, entry in data.items():
        ranked = screen.filter(pl.col("system") == system).sort(["cv_mae_kw", "model"])
        champion = ranked["model"][0]
        limit = min(TUNE_TRIALS, int(np.prod([len(values) for values in SEARCH[champion].values()])))
        candidates = [{}] + list(ParameterSampler(SEARCH[champion], n_iter=limit, random_state=SEED))
        best = None
        for trial, params in enumerate(candidates):
            error, errors = cv_score(system, entry["folds"], champion, MODEL_COLUMNS[champion], params)
            tuning_rows.append(dict(system=system, model=champion, trial=trial, cv_mae_kw=error,
                                    parameters=json.dumps(params, sort_keys=True), fold_mae=errors))
            if best is None or error < best["cv_mae_kw"]:
                best = dict(model=champion, parameters=params, cv_mae_kw=error)
            log.info("%s tuning trial=%d/%d model=%s MAE=%.6f", system,trial,len(candidates)-1,champion,error)
        entry["tuned"] = best
        entry.pop("folds")
tuning = pl.DataFrame(tuning_rows)
fig = go.Figure()
for system in SYSTEMS:
    trials = tuning.filter(pl.col("system") == system)
    fig.add_scatter(x=trials["trial"].to_list(), y=trials["cv_mae_kw"].cum_min().to_list(),
                    mode="lines+markers", name=system, line=dict(color=BLUE if system=="TEC_48S" else ORANGE))
fig.update_xaxes(title="Parameter trial · zero is the untuned candidate", dtick=1)
fig.update_yaxes(title="Best training-fold MAE so far (kW)")
polish(fig, "Did tuning improve the training-fold result?", "Best-so-far score · test data remains untouched")
for system, entry in data.items():
    report(f"**{system}: {entry['tuned']['model']}** is the tuning candidate. Best training-fold MAE is **{entry['tuned']['cv_mae_kw']:.4f} kW**. Validation, not preference, decides whether to select it.")

> **💡 What this tells us**
>
> **TEC_48S: Autoregressive Ridge** is the tuning candidate. Best training-fold MAE is **0.0063 kW**. Validation, not preference, decides whether to select it.

> **💡 What this tells us**
>
> **IPE_PV: Autoregressive Ridge** is the tuning candidate. Best training-fold MAE is **0.1251 kW**. Validation, not preference, decides whether to select it.

### 🏆 7. Select using the reserved validation period
Refit each candidate on all training rows and score the same validation rows. Seasonal fallbacks make the comparison fair even where historical reference readings are absent.

Choose the lowest validation MAE, including simple baselines. Differences below one watt are treated as a practical tie and favour a simpler method. This rule is stated before seeing test scores. If a baseline wins, we keep it; tuning does not guarantee a useful ML improvement.

In [22]:
TIE_TOLERANCE_KW = .001
validation_rows = []
with stage("validation selection", logger=log):
    for system, entry in data.items():
        fitted, evaluation = entry["splits"]["train"], entry["splits"]["validation"]
        actual = evaluation["target_power_kw"].to_numpy()
        specifications = []
        for name in BASELINES:
            predicted, coverage = benchmark(evaluation, name, system)
            validation_rows.append(dict(system=system, model=name, **scores(actual,predicted),
                                         native_history_pct=coverage, fit_seconds=0.))
            specifications.append(dict(label=name, baseline=True, complexity=BASELINES.index(name)))
        for index, (name, columns) in enumerate(MODEL_COLUMNS.items()):
            started = time.perf_counter()
            pipeline = make_pipeline(name,columns)
            with threadpool_limits(limits=THREADS):
                pipeline.fit(arrays(fitted,columns),fitted["target_power_kw"].to_numpy())
                predicted = predict_model(pipeline,evaluation,columns,system)
            validation_rows.append(dict(system=system,model=name,**scores(actual,predicted),
                                         native_history_pct=100.,fit_seconds=time.perf_counter()-started))
            specifications.append(dict(label=name,baseline=False,model=name,columns=columns,parameters={},complexity=10+index))
            del pipeline
        tuned = entry["tuned"]
        label, columns = tuned["model"] + " · tuned", MODEL_COLUMNS[tuned["model"]]
        pipeline = make_pipeline(tuned["model"], columns, tuned["parameters"])
        started = time.perf_counter()
        with threadpool_limits(limits=THREADS):
            pipeline.fit(arrays(fitted,columns),fitted["target_power_kw"].to_numpy())
            predicted = predict_model(pipeline,evaluation,columns,system)
        validation_rows.append(dict(system=system,model=label,**scores(actual,predicted),
                                     native_history_pct=100.,fit_seconds=time.perf_counter()-started))
        specifications.append(dict(label=label,baseline=False,model=tuned["model"],columns=columns,
                                   parameters=tuned["parameters"],complexity=20))
        searched = {entry["tuned"]["model"]}
        while True:
            results = [row for row in validation_rows if row["system"]==system]
            best_error = min(row["mae_kw"] for row in results)
            allowed = {row["model"] for row in results if row["mae_kw"] <= best_error + TIE_TOLERANCE_KW}
            chosen = min((spec for spec in specifications if spec["label"] in allowed),key=lambda spec:spec["complexity"])
            if chosen["baseline"] or chosen["model"] in searched:
                break
            # Tune a newly leading family using training folds only.
            name, columns = chosen["model"], chosen["columns"]
            folds = time_folds(fitted)
            limit = min(TUNE_TRIALS, int(np.prod([len(values) for values in SEARCH[name].values()])))
            parameters = [{}] + list(ParameterSampler(SEARCH[name], n_iter=limit, random_state=SEED))
            best = None
            for trial, params in enumerate(parameters):
                error, errors = cv_score(system, folds, name, columns, params)
                tuning_rows.append(dict(system=system,model=name,trial=trial,cv_mae_kw=error,
                    parameters=json.dumps(params,sort_keys=True),fold_mae=errors))
                if best is None or error < best["error"]:
                    best = dict(error=error, parameters=params)
            searched.add(name)
            extra = make_pipeline(name,columns,best["parameters"])
            started = time.perf_counter()
            with threadpool_limits(limits=THREADS):
                extra.fit(arrays(fitted,columns),fitted["target_power_kw"].to_numpy())
                extra_prediction = predict_model(extra,evaluation,columns,system)
            label = name + " · tuned"
            validation_rows.append(dict(system=system,model=label,**scores(actual,extra_prediction),
                native_history_pct=100.,fit_seconds=time.perf_counter()-started))
            specifications.append(dict(label=label,baseline=False,model=name,columns=columns,
                parameters=best["parameters"],complexity=20))
            log.info("%s additionally tuned validation-leading family=%s",system,name)
            del extra
        chosen["hyperparameter_search_performed"] = not chosen["baseline"]
        chosen["searched_families"] = sorted(searched)
        entry["selected"] = chosen
        entry["selected_validation"] = next(row for row in results if row["model"]==chosen["label"])
        if not chosen["baseline"]:
            columns = chosen["columns"]
            pipeline = make_pipeline(chosen["model"],columns,chosen["parameters"])
            with threadpool_limits(limits=THREADS):
                pipeline.fit(arrays(fitted,columns),fitted["target_power_kw"].to_numpy())
        entry["explanation_model"] = pipeline
        entry["explanation_columns"] = columns
        log.info("%s validation-selected=%s metrics=%s",system,chosen["label"],entry["selected_validation"])
validation = pl.DataFrame(validation_rows)
tuning = pl.DataFrame(tuning_rows)
fig = make_subplots(rows=1,cols=len(SYSTEMS),subplot_titles=list(SYSTEMS),horizontal_spacing=.15)
for column,system in enumerate(SYSTEMS,1):
    ranked = validation.filter(pl.col("system")==system).sort("mae_kw",descending=True)
    selected_label = data[system]["selected"]["label"]
    fig.add_bar(x=ranked["mae_kw"].to_list(),y=ranked["model"].to_list(),orientation="h",
                marker_color=[PURPLE if name==selected_label else "#94a3b8" for name in ranked["model"]],
                showlegend=False,row=1,col=column)
    fig.update_xaxes(title="Validation MAE (kW)",row=1,col=column)
polish(fig,"Which approach earns the final test?","Purple is the selected approach · all scores use the same validation rows",height=600)
for system,entry in data.items():
    seasonal = validation.filter((pl.col("system")==system) & pl.col("model").is_in(["Past day + fallback", "Past week + fallback"]))
    coverage_note = "; ".join(f"{row['model']}: {row['native_history_pct']:.1f}% native history" for row in seasonal.iter_rows(named=True))
    report(f"**{system}: {entry['selected']['label']}** was selected before test evaluation. Every selected ML family received a training-only parameter search; default settings may still win on validation. XGBoost was compared fairly. "
           f"Seasonal validation coverage — {coverage_note}. Remaining rows use persistence.")

> **💡 What this tells us**
>
> **TEC_48S: Persistence** was selected before test evaluation. Every selected ML family received a training-only parameter search; default settings may still win on validation. XGBoost was compared fairly. Seasonal validation coverage — Past day + fallback: 99.5% native history; Past week + fallback: 99.6% native history. Remaining rows use persistence.

> **💡 What this tells us**
>
> **IPE_PV: Random Forest** was selected before test evaluation. Every selected ML family received a training-only parameter search; default settings may still win on validation. XGBoost was compared fairly. Seasonal validation coverage — Past day + fallback: 100.0% native history; Past week + fallback: 100.0% native history. Remaining rows use persistence.

### 🔎 8. What did the selected ML approach use?
Permutation importance measures how much validation error increases when one input is shuffled. It explains dependence, not cause. Correlated features can share or hide importance, and shuffling is only a diagnostic—not a realistic future dataset.

Use at most 1,500 evenly spaced validation rows. This is the only sampling used here; model fitting and headline metrics use every eligible row. For an ML winner, this explains its training-fitted version. If a baseline wins, this chart explains the initial tuned ML challenger rather than pretending the baseline learned feature importance.

In [23]:
fig = make_subplots(rows=1,cols=len(SYSTEMS),subplot_titles=list(SYSTEMS),horizontal_spacing=.2)
importance_rows=[]
with stage("validation feature importance",logger=log):
    for column,(system,entry) in enumerate(data.items(),1):
        evaluation=entry["splits"]["validation"]
        indices=np.unique(np.linspace(0,evaluation.height-1,min(1500,evaluation.height)).astype(int))
        sampled=evaluation[indices.tolist()]
        columns=entry["explanation_columns"]
        with threadpool_limits(limits=THREADS):
            importance=permutation_importance(entry["explanation_model"],arrays(sampled,columns),
                sampled["target_power_kw"].to_numpy(),scoring=lambda estimator, inputs, target: -mean_absolute_error(target,
                    np.maximum(estimator.predict(inputs),0) if system=="IPE_PV" else estimator.predict(inputs)),n_repeats=3,
                n_jobs=1,random_state=SEED)
        ranking=np.argsort(importance.importances_mean)[-8:]
        fig.add_bar(x=importance.importances_mean[ranking].tolist(),y=[columns[i] for i in ranking],
                    orientation="h",marker_color=BLUE if system=="TEC_48S" else ORANGE,
                    showlegend=False,row=1,col=column)
        importance_rows.extend(dict(system=system,feature=feature,mae_increase_kw=float(value))
                               for feature,value in zip(columns,importance.importances_mean))
        entry.pop("explanation_model")
        fig.update_xaxes(title="Validation MAE increase (kW)",row=1,col=column)
polish(fig,"Which inputs helped the selected ML approach?","Diagnostic importance · negative values mean no demonstrated benefit",height=500)
report("Recent power often contains useful short-term information. Weather and calendar effects need enough observed variation; limited solar seasons cannot establish annual behaviour. Importance does not prove that electricity prices cause demand.")

> **💡 What this tells us**
>
> Recent power often contains useful short-term information. Weather and calendar effects need enough observed variation; limited solar seasons cannot establish annual behaviour. Importance does not prove that electricity prices cause demand.

### 🔒 9. Final fit and one untouched test evaluation
The model choice is now frozen. Refit the chosen ML pipeline on train plus validation, then evaluate it once on the existing test split. Baselines use observations available at each forecast origin.

The test is a **rolling short-horizon evaluation**: later forecasts can use newly observed past power. It is not a fixed 24-hour-ahead prediction made once at midnight. We retain all peaks and gaps in eligibility; no test-driven parameter adjustments follow.

In [24]:
test_rows=[]
with stage("final refit and held-out test",logger=log):
    for system,entry in data.items():
        specification=entry["selected"]
        evaluation=entry["splits"]["test"]
        development=pl.concat([entry["splits"]["train"],entry["splits"]["validation"]]).sort("WsDateTime")
        if specification["baseline"]:
            model=None
            prediction,_=benchmark(evaluation,specification["label"],system)
        else:
            model=make_pipeline(specification["model"],specification["columns"],specification["parameters"])
            with threadpool_limits(limits=THREADS):
                model.fit(arrays(development,specification["columns"]),development["target_power_kw"].to_numpy())
                prediction=predict_model(model,evaluation,specification["columns"],system)
        persistence,_=benchmark(evaluation,"Persistence",system)
        actual=evaluation["target_power_kw"].to_numpy()
        selected_metrics,reference_metrics=scores(actual,prediction),scores(actual,persistence)
        improvement=(100*(1-selected_metrics["mae_kw"]/reference_metrics["mae_kw"])
                     if reference_metrics["mae_kw"] else None)
        entry["model"],entry["test_metrics"],entry["skill_pct"]=model,selected_metrics,improvement
        entry["predictions"]=evaluation.select("WsDateTime","Target_Start","Target_End").with_columns(
            pl.Series("actual_kw",actual),pl.Series("predicted_kw",prediction),pl.Series("persistence_kw",persistence)
        ).with_columns((pl.col("predicted_kw")-pl.col("actual_kw")).alias("error_kw"),
                       pl.col("Target_Start").dt.convert_time_zone(TIMEZONE).dt.date().alias("Date"))
        for label,metrics in [(specification["label"],selected_metrics),("Persistence reference",reference_metrics)]:
            test_rows.append(dict(system=system,model=label,**metrics))
        cards([
            (f"{system} selected",specification["label"],"choice frozen before test scoring"),
            ("Test MAE",f"{selected_metrics['mae_kw']:.4f} kW","average absolute power error"),
            ("Test RMSE",f"{selected_metrics['rmse_kw']:.4f} kW","larger errors receive more weight"),
            ("Skill vs persistence",f"{improvement:.1f}%" if improvement is not None else "Undefined","positive means lower MAE"),
            ("Bias",f"{selected_metrics['bias_kw']:+.4f} kW","positive means overprediction"),
            ("Test windows",f"{evaluation.height:,}","all eligible held-out windows"),
        ])
        log.info("%s test=%s persistence=%s skill_pct=%s",system,selected_metrics,reference_metrics,improvement)
test_results=pl.DataFrame(test_rows)
report("A negative skill score means the selected method did not beat persistence on later data. That result must be kept. WAPE is secondary because near-zero demand can make percentage errors unstable; ordinary MAPE is deliberately omitted.")

> **💡 What this tells us**
>
> A negative skill score means the selected method did not beat persistence on later data. That result must be kept. WAPE is secondary because near-zero demand can make percentage errors unstable; ordinary MAPE is deliberately omitted.

### 📈 10. See the forecast across the held-out period
The first chart uses daily mean power on test dates. It is an overview of eligible windows, not full-day energy or a complete calendar-day average. Missing dates break the lines.

The second chart shows the best-covered test date with positive measured power, when one exists; otherwise it uses the best-covered date. Both actual and predicted values are quarter-hour averages. This example is selected for readable coverage, not because its errors look favourable.

In [25]:
for system,entry in data.items():
    predictions=entry["predictions"]
    daily=predictions.group_by("Date").agg(pl.len().alias("windows"),
        pl.col("actual_kw").mean(),pl.col("predicted_kw").mean(),pl.col("persistence_kw").mean(),
        pl.col("error_kw").abs().mean().alias("mae_kw"),pl.col("actual_kw").max().alias("peak_kw")).sort("Date")
    calendar=pl.DataFrame({"Date":pl.date_range(daily["Date"].min(),daily["Date"].max(),interval="1d",eager=True)})
    shown=calendar.join(daily,on="Date",how="left")
    fig=go.Figure()
    for column,label,color,dash in [("actual_kw","Measured",BLUE,"solid"),("predicted_kw","Selected forecast",ORANGE,"dash")]:
        fig.add_scatter(x=shown["Date"].cast(pl.String).to_list(),y=shown[column].to_list(),name=label,
                        mode="lines",line=dict(color=color,width=2,dash=dash),connectgaps=False)
    fig.update_xaxes(title="Berlin calendar date")
    fig.update_yaxes(title="Mean power of eligible test windows (kW)")
    polish(fig,f"{system}: forecast through the held-out period","Daily mean over matched eligible windows · missing dates remain gaps")
    report("These lines summarize the same windows. An incomplete day is not scaled up to a full day, and excluded target intervals remain outside the score.")
    eligible=daily.filter(pl.col("peak_kw")>0)
    example=(eligible if eligible.height else daily).sort(["windows","Date"],descending=[True,False])["Date"][0]
    selected=predictions.filter(pl.col("Date")==example).sort("Target_Start")
    # Keep absent quarter-hours visible instead of drawing a bridge over gaps.
    start=datetime.combine(example,datetime.min.time()).replace(tzinfo=ZoneInfo(TIMEZONE))
    end=datetime.combine(example+timedelta(days=1),datetime.min.time()).replace(tzinfo=ZoneInfo(TIMEZONE))
    full_day=pl.DataFrame({"Target_Start":pl.datetime_range(start.astimezone(timezone.utc),end.astimezone(timezone.utc),interval="15m",closed="left",eager=True)}).with_columns(pl.col("Target_Start").cast(pl.Datetime("us","UTC")))
    detail=full_day.join(selected,on="Target_Start",how="left")
    fig=go.Figure()
    for column,label,color,dash in [("actual_kw","Measured",BLUE,"solid"),("predicted_kw","Selected forecast",ORANGE,"solid"),("persistence_kw","Persistence",PURPLE,"dash")]:
        fig.add_scatter(x=clock(detail["Target_Start"]),y=detail[column].to_list(),name=label,mode="lines",
                        line=dict(color=color,width=2,shape="hv",dash=dash),connectgaps=False)
    fig.update_xaxes(title="Berlin local time",tickformat="%H:%M")
    fig.update_yaxes(title="Quarter-hour average power (kW)")
    polish(fig,f"{system}: one test day — {example:%d %B %Y}","Best-covered eligible date · forecast uses only earlier observations")
    report(f"This date contains **{selected.height} scored intervals**. Short peaks remain difficult; five-second and instantaneous peaks are not the forecasting target here.")
    entry["daily_diagnostics"]=daily

> **💡 What this tells us**
>
> These lines summarize the same windows. An incomplete day is not scaled up to a full day, and excluded target intervals remain outside the score.

> **💡 What this tells us**
>
> This date contains **96 scored intervals**. Short peaks remain difficult; five-second and instantaneous peaks are not the forecasting target here.

> **💡 What this tells us**
>
> These lines summarize the same windows. An incomplete day is not scaled up to a full day, and excluded target intervals remain outside the score.

> **💡 What this tells us**
>
> This date contains **96 scored intervals**. Short peaks remain difficult; five-second and instantaneous peaks are not the forecasting target here.

### ⚠️ 11. Where does the model struggle?
Compare absolute error in lower- and higher-power intervals. The boundary is the 75th percentile of training targets; test data never chooses the threshold. A group may be empty or small, so counts are shown.

Daily signed error also shows persistent over- or underprediction. These diagnostics identify weaknesses; they do not authorize tuning on the test period.

In [26]:
fig=make_subplots(rows=1,cols=len(SYSTEMS),subplot_titles=list(SYSTEMS),horizontal_spacing=.15)
for column,(system,entry) in enumerate(data.items(),1):
    threshold=float(entry["splits"]["train"]["target_power_kw"].quantile(.75))
    predictions=entry["predictions"]
    labels,values,counts=[],[],[]
    for label,condition in [("Lower power",pl.col("actual_kw")<=threshold),("Higher power",pl.col("actual_kw")>threshold)]:
        group=predictions.filter(condition)
        labels.append(label);counts.append(group.height)
        values.append(float(group["error_kw"].abs().mean()) if group.height else None)
    fig.add_bar(x=labels,y=values,marker_color=[BLUE,RED],showlegend=False,
                customdata=counts,text=[f"n={count:,}" for count in counts],textposition="outside",
                hovertemplate="%{x}<br>MAE %{y:.4f} kW<br>%{customdata:,} windows<extra></extra>",row=1,col=column)
    fig.update_yaxes(title="Test MAE (kW)",row=1,col=column)
polish(fig,"Does error grow when power is higher?","Power bands fixed from training data · test is diagnostic only")
report("A low overall error can hide poor performance during rare high-demand events. Scheduling decisions should consider peak errors, bias and coverage, not only a single average score.")

> **💡 What this tells us**
>
> A low overall error can hide poor performance during rare high-demand events. Scheduling decisions should consider peak errors, bias and coverage, not only a single average score.

### 💾 12. Save a reproducible result
When `SAVE_ARTIFACTS=True`, results are written to a new timestamped folder under `models/forecasting/`. The ETL datasets remain unchanged.

Each system gets its selected pipeline (when ML wins), a prediction file, scores, parameter-search results and a recipe explaining features, baseline fallback, units and source hashes. A baseline winner is saved as a recipe rather than a misleading empty model.

Optional MLflow tracking is off by default. Enable it only when ready; it uses a local SQLite backend. The exported pipeline expects the ordered engineered features described in its recipe—this notebook does not yet implement a complete live feature-serving API.

In [27]:
def sha256_file(path):
    digest=hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda:handle.read(1024*1024),b""):
            digest.update(block)
    return digest.hexdigest()

def write_json(path,payload):
    path.write_text(json.dumps(payload,indent=2,allow_nan=False,default=str),encoding="utf-8")

versions={name:metadata.version(name) for name in ["numpy","polars","pyarrow","scikit-learn","plotly","joblib"]}
versions["xgboost"]=__import__('xgboost').__version__
versions["python"]=platform.python_version()
if SAVE_ARTIFACTS:
    OUTPUT.mkdir(parents=True,exist_ok=False)
    with stage("save forecasting artifacts",logger=log):
        screen.write_parquet(OUTPUT/"training_cv.parquet")
        tuning.write_parquet(OUTPUT/"tuning_trials.parquet")
        validation.write_parquet(OUTPUT/"validation_scores.parquet")
        test_results.write_parquet(OUTPUT/"test_scores.parquet")
        pl.DataFrame(importance_rows).write_parquet(OUTPUT/"validation_importance.parquet")
        for system,entry in data.items():
            folder=OUTPUT/system;folder.mkdir()
            entry["predictions"].write_parquet(folder/"test_predictions.parquet")
            entry["daily_diagnostics"].write_parquet(folder/"test_daily_diagnostics.parquet")
            if entry["model"] is not None:
                joblib.dump(entry["model"],folder/"model.joblib",compress=3)
            recipe=dict(system=system,run_id=RUN_ID,selection=entry["selected"],settings=settings,
                validation_metrics=entry["selected_validation"],test_metrics=entry["test_metrics"],
                test_skill_pct=entry["skill_pct"],versions=versions,target_unit="kW",interval_minutes=15,
                prediction_origin="end of latest observed quarter-hour; Target_Start equals origin",
                history_policy="exact elapsed-UTC-day seasonal joins; no target filling",
                baseline_missing_history="fall back to latest observed quarter-hour power",
                feature_order=entry["selected"].get("columns",[]),
                forecast_scope="rolling next interval; not next-day multistep",
                calendar_timezone=TIMEZONE,source_sha256={name:sha256_file(path) for name,path in entry["paths"].items()},
                training_target_coverage=entry["quality"]["features"],
                availability_assumptions=["previous-hour DWD uses ETL delay", "imminent SMARD day-ahead price assumed published"],
                solar_lower_bound=0.0 if system=="IPE_PV" else None,
                eligible_test_rows=entry["splits"]["test"].height)
            write_json(folder/"recipe.json",recipe)
        write_json(OUTPUT/"run.json",dict(run_id=RUN_ID,settings=settings,versions=versions,systems=list(SYSTEMS)))
    report(f"Saved this run under **models/forecasting/{RUN_ID}/**. Existing ETL data and earlier model runs are unchanged.")
else:
    report("Artifact export is disabled. Results remain in notebook memory for this session.")

if ENABLE_MLFLOW:
    if not SAVE_ARTIFACTS:
        raise ValueError("Enable artifact saving before enabling MLflow.")
    try:
        import mlflow
    except ImportError as error:
        raise ImportError("Install mlflow in this environment or keep ENABLE_MLFLOW=False.") from error
    database=PROJECT/"mlflow.db"
    mlflow.set_tracking_uri("sqlite:///"+database.as_posix())
    mlflow.set_experiment("GreenForge_quarter_hour_forecasting")
    with stage("MLflow experiment tracking",logger=log):
        for system,entry in data.items():
            with mlflow.start_run(run_name=f"{system}_{RUN_ID}"):
                mlflow.log_params(dict(system=system,selected=entry["selected"]["label"],**settings))
                mlflow.log_metrics({key:value for key,value in entry["test_metrics"].items() if value is not None})
                mlflow.set_tags(dict(etl_quality="target_gate_passed",forecast_scope="rolling_next_15_minutes"))
                mlflow.log_artifacts(str(OUTPUT/system),artifact_path=system)
                mlflow.log_artifact(str(OUTPUT/"validation_scores.parquet"))

> **💡 What this tells us**
>
> Saved this run under **models/forecasting/20261003_164052_015558Z/**. Existing ETL data and earlier model runs are unchanged.

### 🏁 13. Answers and the next step
The conclusions below are generated from your actual run. No model winner or savings figure is hard-coded. Model selection uses validation; test results describe how that fixed choice performed later.

In [28]:
for system,entry in data.items():
    score=entry["test_metrics"]
    skill=entry["skill_pct"]
    performance=(f"It reduced test MAE by **{skill:.1f}%** relative to persistence." if skill is not None and skill>0
                 else "It did not demonstrate lower test MAE than persistence; retain that result and investigate without retuning on this test.")
    display(Markdown(f"""
### {'🏭' if system=='TEC_48S' else '☀️'} {system}
**Which approach was selected?** {entry['selected']['label']}, chosen before seeing test errors.

**How accurate was it?** Test MAE is **{score['mae_kw']:.4f} kW**, RMSE **{score['rmse_kw']:.4f} kW**, and signed bias **{score['bias_kw']:+.4f} kW**. {performance}

**What can we claim?** Performance on the published eligible windows for a rolling quarter-hour forecast. Missing source periods are not reconstructed. Solar results do not establish performance in seasons without observations.
"""))
report("**Scheduling:** these short-horizon models are a starting point. A next-day scheduler needs a separate multi-horizon design and feasible job constraints.\n\n"
       "**Sustainability:** forecast machine demand and solar first. Only justified, aligned grid-import energy should be converted to a year-matched UBA estimate. A constant annual factor cannot identify cleaner hours.\n\n"
       "**Deployment:** move the tested feature recipe, validation and fitting functions into the existing Python modules, add contract tests, and retain the saved data hashes and model settings. Forecast accuracy alone does not establish factory-wide savings or safe machine operation.")
log.info("Forecasting notebook complete: run=%s",RUN_ID)


### 🏭 TEC_48S
**Which approach was selected?** Persistence, chosen before seeing test errors.

**How accurate was it?** Test MAE is **0.0059 kW**, RMSE **0.0711 kW**, and signed bias **-0.0003 kW**. It did not demonstrate lower test MAE than persistence; retain that result and investigate without retuning on this test.

**What can we claim?** Performance on the published eligible windows for a rolling quarter-hour forecast. Missing source periods are not reconstructed. Solar results do not establish performance in seasons without observations.



### ☀️ IPE_PV
**Which approach was selected?** Random Forest, chosen before seeing test errors.

**How accurate was it?** Test MAE is **0.1511 kW**, RMSE **0.4246 kW**, and signed bias **-0.0160 kW**. It reduced test MAE by **3.5%** relative to persistence.

**What can we claim?** Performance on the published eligible windows for a rolling quarter-hour forecast. Missing source periods are not reconstructed. Solar results do not establish performance in seasons without observations.


> **💡 What this tells us**
>
> **Scheduling:** these short-horizon models are a starting point. A next-day scheduler needs a separate multi-horizon design and feasible job constraints.
> 
> **Sustainability:** forecast machine demand and solar first. Only justified, aligned grid-import energy should be converted to a year-matched UBA estimate. A constant annual factor cannot identify cleaner hours.
> 
> **Deployment:** move the tested feature recipe, validation and fitting functions into the existing Python modules, add contract tests, and retain the saved data hashes and model settings. Forecast accuracy alone does not establish factory-wide savings or safe machine operation.

### 📚 Method references
* [Scikit-learn: avoiding leakage with preprocessing pipelines](https://scikit-learn.org/stable/common_pitfalls.html).
* [Scikit-learn: histogram gradient boosting](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.HistGradientBoostingRegressor.html). Random internal early stopping is disabled here.
* [XGBoost: Python estimator parameters](https://xgboost.readthedocs.io/en/stable/python/python_api.html). This notebook selects tree count through temporal parameter search instead of using the held-out test as an early-stopping set.

This notebook is supplied without fabricated training outputs. Run all cells on your published datasets to obtain the real winner and metrics.